# 03 — Exploratory Data Analysis

**Owner:** Analytics Lead
**Input:** `data/processed/online_retail_II_clean.csv`
**Output:** Written insights + charts (inline); feeds both the dashboard and the report
**Rubric link:** Analysis Depth (25 marks)

## Framing rule
Every chart gets a **written insight** in a markdown cell directly below it. Chart without insight = zero rubric value. Insight = "X is Y% of Z because of W" — decision language, not description.

## EDA plan (mapped to sub-questions)

1. **Revenue shape** — monthly trend, top countries, UK share
2. **Pareto check** — what % of revenue comes from top 20% of customers? (sub-question 1)
3. **Guest vs registered** — AOV and basket composition differences (sub-question 2)
4. **Product category performance** — revenue + return rate by category (sub-question 3)
5. **Seasonality** — monthly pattern; hour-of-day + day-of-week (sub-question 4)
6. **Retention** — repeat-purchase rate over time; cohort heatmap
7. **Outliers** — unusually large orders, unusual customers


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_theme(style="whitegrid", context="notebook")
df = pd.read_csv("../data/processed/online_retail_II_clean.csv", parse_dates=["invoicedate"])
print(df.shape)
df.head()

### 1. Monthly revenue trend

In [ ]:
monthly = (df[df["is_product"]]
           .groupby("invoice_year_month")["line_revenue"].sum()
           .sort_index())
monthly.plot(kind="line", marker="o", figsize=(10,4), title="Monthly Net Revenue")
plt.ylabel("Revenue (GBP)")
plt.tight_layout()
plt.show()
monthly.describe()

> **Insight:** *(write 1–2 sentences describing the trend and what it means for the business)*

### 2. Top-20% customer revenue share (Pareto)

In [ ]:
reg = df[df["is_registered"] & df["is_product"]]
cust_rev = reg.groupby("customer_id")["line_revenue"].sum().sort_values(ascending=False)
top20 = cust_rev.head(int(len(cust_rev)*0.20)).sum()
total = cust_rev.sum()
print(f"Top 20% of customers drive {top20/total*100:.1f}% of revenue (n={len(cust_rev)} registered customers)")

> **Insight:** *(interpret the concentration — is it classic 80/20 or more extreme?)*

### 3. Guest vs Registered AOV

In [ ]:
# AOV = revenue per invoice. Compare guest vs registered.
invoice_level = (df[df["is_product"] & ~df["is_return"]]
                 .groupby(["invoice","is_registered"])["line_revenue"].sum()
                 .reset_index())
invoice_level.groupby("is_registered")["line_revenue"].describe()

In [ ]:
sns.boxplot(data=invoice_level, x="is_registered", y="line_revenue", showfliers=False)
plt.title("Order value: registered vs guest")
plt.ylabel("Invoice revenue (GBP)")
plt.show()

> **Insight:** *(is the difference material? A formal test will follow in notebook 04)*

### 4. Return rate by category hint

In [ ]:
cat_stats = (df[df["is_product"]]
             .groupby("product_category_hint")
             .agg(rows=("invoice","size"),
                  returns=("is_return","sum"),
                  revenue=("line_revenue","sum")))
cat_stats["return_rate"] = cat_stats["returns"] / cat_stats["rows"]
cat_stats.sort_values("revenue", ascending=False)

> **Insight:** *(which categories over- or under-index on returns?)*

### 5. Seasonality — hour of day & day of week

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14,4))
df.groupby("invoice_hour")["line_revenue"].sum().plot(kind="bar", ax=axes[0], title="Revenue by hour")
dow_order = ["Monday","Tuesday","Wednesday","Thursday","Friday","Saturday","Sunday"]
df.groupby("invoice_day_of_week")["line_revenue"].sum().reindex(dow_order).plot(kind="bar", ax=axes[1], title="Revenue by day of week")
plt.tight_layout()

> **Insight:** *(pinpoint peak hours/days; operational implication)*

### 6. Cohort retention heatmap

In [ ]:
reg = df[df["is_registered"] & df["is_product"] & ~df["is_return"]].copy()
reg["order_month"]  = reg["invoicedate"].dt.to_period("M")
reg["cohort_month"] = reg.groupby("customer_id")["invoicedate"].transform("min").dt.to_period("M")
reg["cohort_index"] = ((reg["order_month"] - reg["cohort_month"]).apply(lambda x: x.n))

cohort = (reg.groupby(["cohort_month","cohort_index"])["customer_id"]
          .nunique().unstack(0).T)
cohort_pct = cohort.divide(cohort[0], axis=0)
plt.figure(figsize=(12,8))
sns.heatmap(cohort_pct*100, annot=False, fmt=".0f", cmap="YlGnBu")
plt.title("Cohort retention (% of month-0 customers retained)")
plt.ylabel("Cohort month"); plt.xlabel("Months since first purchase")
plt.show()

> **Insight:** *(describe retention curve shape; point to segments with steepest drop-off)*

## Summary of EDA findings

*(After running all cells, write 5–7 bullet observations here. These become candidate insights for the final report.)*